# 15 – Phase 5a: noisy labels (training-side filter, validation unchanged)

**Question:** does training *without* the images whose label is probably wrong improve the model?

**What we know so far:**
- In the EDA, 126 training images were flagged `review`: a simple model (linear probe) confidently disagreed with their label.
- Fatemeh's V4 agrees with those labels only ~44% of the time (88% elsewhere). Part of this is by construction, though: the images were *chosen* because a model disagreed.
- Her "0.859 → 0.879" only **excluded them from the evaluation**, so it is not a measured gain.

The real effect of cleaning the *training* set has never been measured. That is what this notebook does.

**Two rules to keep the measurement honest:**
1. **The validation never changes.** Every run is evaluated on all 2,995 images with their original labels, on both validations, exactly like every previous run. If a validation label is wrong, a better model is *penalised* on it, so this measurement can only be pessimistic, never inflated.
2. **No information from the validation fold reaches the filter.** For each outer fold k, the images to drop are chosen by a cross-validation **inside its training part only** (nested CV):
   - the 2,396 training images of fold k are split into 5 inner folds;
   - a model trained on 4 of them predicts the 5th;
   - so every training image gets an out-of-fold probability from a model that never saw it, nor any image of the validation fold k.

   The EDA `review` flags cannot be used directly, because they were computed with *all* images, including the validation images of each fold.

**The flagger.** It is the reference ConvNeXt-Tiny, trained shorter to save time: 12 epochs, batch 64. This is enough, because the baseline plateaued around epoch 10–12. It is much stronger than the EDA linear probe (OOF F1 ≈ 0.85 vs ≈ 0.62), so its disagreements are more meaningful.

An image is **flagged** when the model **confidently** disagrees:
- probability of the given label < 0.1;
- probability of the predicted class > 0.6.

It costs 5 outer × 5 inner = 25 short trainings (≈ 30 min), computed once and cached.

**Runs** (everything else identical to `ref_256` of notebook 14):

| run | flagged training images | idea |
|---|---|---|
| `ref_256` | kept (weight 1), already trained | reference |
| `lbl_drop` | removed (weight 0) | a wrong label teaches the wrong thing: remove it |
| `lbl_down` | kept with weight 0.3 | softer: if the flag is wrong (a hard but correct image), it is not lost entirely |

Relabelling flagged images with the predicted class is **not** tested: if the flagger is wrong, the model would learn the error with full confidence.

**Decision rule:** as in notebook 14, adopt only if the OOF F1 rises by **≳ 0.01**, the clean validation does not drop, and cigar does not collapse. The run-to-run noise is up to ±0.02 on a single fold and a few thousandths on the OOF.

**Time:** ≈ 30 min (flagger) + 2 × 16 min ≈ **1 h 05 min**. Run it in `tmux` from the VS Code terminal:
```
tmux new -s phase5a
cd ~/ANN-DL-Challenge/families/elia && source ../../.venv/bin/activate
jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 15_phase5a_label_noise.ipynb
```

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

import train_lib as T
import common as C

plt.rcParams["figure.dpi"] = 80
SMOKE_TEST = False          # True = tiny run (few images, 1 epoch) only to check that the code works
data = T.load_data(smoke_test=SMOKE_TEST)
REF = "ref_256"             # trained in notebook 14
print("device:", T.DEVICE, "| outputs:", T.OUT_DIR)
assert SMOKE_TEST or (T.OUT_DIR / REF / "oof.csv").exists(), "run notebook 14 first (ref_256)"

## 1. Find the likely wrong labels (nested cross-validation)

`T.find_label_issues` trains 25 short models and saves `outputs/label_flags/flags.csv`. The file has one row per (outer fold, training image), with:
- the given label;
- the predicted class;
- the two probabilities;
- the flag.

Each image appears in 4 outer folds (it is a training image in all folds except its own), and can be flagged in some of them and not in others.

In [ ]:
FLAG_NAME = "label_flags_SMOKE" if SMOKE_TEST else "label_flags"
flag_cfg = T.reference_config(run_name=FLAG_NAME, epochs=1 if SMOKE_TEST else 12, batch_size=16 if SMOKE_TEST else 64)
flags = T.find_label_issues(flag_cfg, data, name=FLAG_NAME)
FLAGS_PATH = str(T.OUT_DIR / FLAG_NAME / "flags.csv")

## 2. What the filter found

How many images are flagged, in which classes, and towards which predicted class? Overlap with the EDA `review` flags: they come from a different, weaker model on all the data, so a large overlap means that two independent models agree that those labels are suspicious.

In [ ]:
per_fold = flags.groupby("outer_fold").agg(training_images=("id", "size"), flagged=("flagged", "sum"))
per_fold["inner_oof_accuracy"] = (flags.pred == flags.label).groupby(flags.outer_fold).mean()
per_fold["flagged %"] = 100 * per_fold.flagged / per_fold.training_images
display(per_fold.round(3))

# an image is "ever flagged" if flagged in at least one outer fold, "always flagged" if in all its outer folds
img = flags.groupby("id").agg(label=("label", "first"), times=("flagged", "sum"), n=("flagged", "size"))
ever, always = img[img.times > 0], img[img.times == img.n]
review_ids = set(data["folds"].id[data["folds"].review.astype(bool)])
print(f"images flagged in at least one outer fold: {len(ever)}  | in all of their outer folds: {len(always)}")
print(f"EDA review images: {len(review_ids)}  | of which flagged at least once here: {len(review_ids & set(ever.index))}")

In [ ]:
# from which class (rows, given label) to which class (columns, predicted) the flagged images go
fl = flags[flags.flagged].drop_duplicates("id")
names = [f"{c} {C.CLASSES[c]}" for c in range(C.N_CLASSES)]
trans = pd.crosstab(fl.label.map(dict(enumerate(names))), fl.pred.map(dict(enumerate(names))))
display(trans)
share = (img.assign(f=img.times > 0).groupby("label").f.mean() * 100).rename(index=dict(enumerate(names)))
print("share of each class flagged at least once (%):"); print(share.round(1).to_string())

### Visual check

24 flagged images: the title shows the **given label → predicted class (its probability)**.

**Check by eye:** in how many of them do you agree with the model rather than with the label? If most look like genuinely wrong labels, the filter is doing its job. If they look like correct but hard or ambiguous galaxies, dropping them could hurt, because the model would lose exactly the hard examples it needs.

In [ ]:
show = fl.sort_values("p_pred", ascending=False).head(24)
pos = pd.Series(np.arange(len(data["folds"])), index=data["folds"].id)
fig, axes = plt.subplots(3, 8, figsize=(17, 7.2))
for ax, (_, r) in zip(axes.ravel(), show.iterrows()):
    ax.imshow(data["X_clean"][pos[r.id]]); ax.axis("off")
    ax.set_title(f"{C.CLASSES[r.label][:12]} → {C.CLASSES[r.pred][:12]} ({r.p_pred:.2f})", fontsize=8)
for ax in axes.ravel()[len(show):]:
    ax.axis("off")
plt.suptitle("Flagged training images: given label → predicted class (probability)"); plt.tight_layout(); plt.show()

## 3. Train without (or down-weighting) the flagged images

In [ ]:
EXPERIMENTS = [
    T.reference_config(run_name="lbl_drop", note="ref_256 - flagged labels removed from training (nested CV)",
                       label_flags=FLAGS_PATH, flag_weight=0.0),
    T.reference_config(run_name="lbl_down", note="ref_256 - flagged labels down-weighted to 0.3 (nested CV)",
                       label_flags=FLAGS_PATH, flag_weight=0.3),
]
if SMOKE_TEST:
    for cfg in EXPERIMENTS:
        cfg.update(run_name=cfg["run_name"] + "_SMOKE", epochs=1, folds=[0], batch_size=16)
results = {}
for cfg in EXPERIMENTS:
    results[cfg["run_name"]] = T.run_experiment(cfg, data)

## 4. Comparison with `ref_256` (full validation, original labels)

This is the comparison that decides: same 2,995 validation images, same labels, both validations.

In [ ]:
RUNS = [cfg["run_name"] for cfg in EXPERIMENTS]
table = T.compare(RUNS, reference=REF)
table.style.format("{:.4f}").background_gradient(subset=["Δ clean", "Δ shifted"], cmap="RdYlGn", vmin=-0.02, vmax=0.02)

In [ ]:
# fold by fold (paired, same validation images): Δ = run − ref_256
def fold_f1(run, mask=None):
    oof = pd.read_csv(T.OUT_DIR / run / "oof.csv")
    m = np.ones(len(oof), bool) if mask is None else mask
    out = {}
    for val in ["clean", "shifted"]:
        P = oof[[f"p{c}_{val}" for c in range(C.N_CLASSES)]].to_numpy()
        out[val] = [f1_score(oof.label[(oof.fold == k) & m], P[(oof.fold == k) & m].argmax(1), average="weighted")
                    for k in range(5)]
    return out

ref = fold_f1(REF)
rows = []
for run in RUNS:
    if not (T.OUT_DIR / run / "oof.csv").exists():
        continue
    f = fold_f1(run)
    for val in ["clean", "shifted"]:
        d = np.array(f[val]) - np.array(ref[val])
        rows.append({"run": run, "validation": val, **{f"fold {k}": d[k] for k in range(5)},
                     "mean Δ": d.mean(), "folds better": int((d > 0).sum())})
pd.DataFrame(rows).set_index(["run", "validation"]).style.format(
    {**{f"fold {k}": "{:+.4f}" for k in range(5)}, "mean Δ": "{:+.4f}"}
).background_gradient(subset=["mean Δ"], cmap="RdYlGn", vmin=-0.02, vmax=0.02)

In [ ]:
names_runs = [REF] + RUNS
for val in ["clean", "shifted"]:
    t = T.per_class(names_runs, val)
    d = t.sub(t[REF], axis=0).drop(columns=REF)
    print(f"{val.upper()} validation: per-class F1 of {REF}, then Δ")
    display(pd.concat([t[[REF]], d], axis=1).style.format("{:+.3f}").format("{:.3f}", subset=[REF])
            .background_gradient(subset=list(d.columns), cmap="RdYlGn", vmin=-0.05, vmax=0.05))

## 5. For information only: F1 on the "unsuspicious" validation images

The same OOF predictions, scored after **excluding from the evaluation** the validation images whose labels are suspicious (the EDA `review` images). This is **not** used for the decision. It removes hard images from the calculation for every model alike, which inflates all the scores.

It only answers a side question: is the gain or loss of the filtered models concentrated on the doubtful images, or spread over the normal ones?

In [ ]:
oof0 = pd.read_csv(T.OUT_DIR / REF / "oof.csv")
normal = ~oof0.id.isin(review_ids).to_numpy()
rows = []
for run in [REF] + RUNS:
    if not (T.OUT_DIR / run / "oof.csv").exists():
        continue
    oof = pd.read_csv(T.OUT_DIR / run / "oof.csv")
    for val in ["clean", "shifted"]:
        P = oof[[f"p{c}_{val}" for c in range(C.N_CLASSES)]].to_numpy().argmax(1)
        rows.append({"run": run, "validation": val,
                     "F1 all images": f1_score(oof.label, P, average="weighted"),
                     "F1 without review images": f1_score(oof.label[normal], P[normal], average="weighted"),
                     "accuracy on review images": (oof.label[~normal] == P[~normal]).mean()})
pd.DataFrame(rows).set_index(["run", "validation"]).round(4)

## 6. Decision

Rule: adopt if the OOF F1 rises by ≳ 0.01, the clean validation does not drop, and cigar does not collapse.

| run | F1 clean (Δ) | F1 shifted (Δ) | folds better (clean / shifted) | verdict |
|---|---|---|---|---|
| lbl_drop | ___ | ___ | ___ | ___ |
| lbl_down | ___ | ___ | ___ | ___ |

- Flagged images per fold: ___; overlap with the EDA `review` images: ___
- Visual check (section 2): do the flagged images look like wrong labels? ___
- **Decision:** ___